В данном ноутбуке мы будем получать через api superjob 

In [2]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [87]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='../../.env')

SUPERJOB_SECRET = os.getenv('SUPERJOB_SECRET_KEY')
SUPERJOB_CLIENT_ID = os.getenv('SUPERJOB_CLIENT_ID')
SUPERJOB_LOGIN = os.getenv('SUPERJOB_LOGIN')
SUPERJOB_PASSWORD = os.getenv('SUPERJOB_PASSWORD')

In [88]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

In [89]:
# Авторизуемся

params = {
    'client_id': SUPERJOB_CLIENT_ID,
    'client_secret': SUPERJOB_SECRET,
    'login': SUPERJOB_LOGIN,
    'password': SUPERJOB_PASSWORD,
}

auth_response = requests.post('https://api.superjob.ru/2.0/oauth2/password/', params=params)

In [90]:
token = auth_response.json()['access_token']
headers = {
    "Authorization": f"Bearer {token}",
    "X-Api-App-Id": SUPERJOB_SECRET
}

In [91]:
industry = requests.get('https://api.superjob.ru/2.0/catalogues/', headers=headers)
industry_res = industry.json()
industry_res[0]

{'title_rus': 'IT, Интернет, связь, телеком',
 'url_rus': 'it-internet-svyaz-telekom',
 'title': 'IT, Интернет, связь, телеком',
 'title_trimmed': 'IT, Интернет, связь,...',
 'key': 33,
 'positions': [{'title_rus': 'AI',
   'url_rus': 'ai',
   'title': 'AI',
   'id_parent': 33,
   'key': 651},
  {'title_rus': 'CRM-системы',
   'url_rus': 'crm-sistemy',
   'title': 'CRM-системы',
   'id_parent': 33,
   'key': 603},
  {'title_rus': 'Data Science',
   'url_rus': 'data-science',
   'title': 'Data Science',
   'id_parent': 33,
   'key': 627},
  {'title_rus': 'DevOps',
   'url_rus': 'devops',
   'title': 'DevOps',
   'id_parent': 33,
   'key': 628},
  {'title_rus': 'SRE',
   'url_rus': 'sre',
   'title': 'SRE',
   'id_parent': 33,
   'key': 629},
  {'title_rus': 'Web-верстка',
   'url_rus': 'web-verstka',
   'title': 'Web-верстка',
   'id_parent': 33,
   'key': 36},
  {'title_rus': 'Администрирование баз данных',
   'url_rus': 'administrirovanie-baz-dannyh',
   'title': 'Администрирование ба

Здесь сразу можно отметить, что 'IT, Интернет, связь, телеком' - первая отрасль и  ее 'key': 33

In [92]:
titles = []
for catalogue in industry_res:
    titles.append(catalogue['title_rus'])
print(titles)

['IT, Интернет, связь, телеком', 'Административная работа, секретариат, АХО', 'Банки, инвестиции, лизинг', 'Безопасность, службы охраны', 'Бухгалтерия, финансы, аудит', 'Государственная служба', 'Дизайн', 'Домашний персонал', 'Закупки, снабжение', 'Искусство, культура, развлечения', 'Кадры, управление персоналом', 'Консалтинг, стратегическое развитие', 'Маркетинг, реклама, PR', 'Медицина, фармацевтика, ветеринария', 'Наука, образование, повышение квалификации', 'Некоммерческие организации, волонтерство', 'Обмен персоналом', 'Продажи', 'Промышленность, производство', 'Рабочий персонал', 'Сельское хозяйство', 'Службы доставки', 'СМИ, издательства', 'Спорт, фитнес, салоны красоты, SPA', 'Страхование', 'Строительство, проектирование, недвижимость', 'Сырье', 'Топ-персонал', 'Транспорт, логистика, ВЭД', 'Туризм, гостиницы, общественное питание', 'Услуги, ремонт, сервисное обслуживание', 'Юриспруденция']


Список отраслей: ['IT, Интернет, связь, телеком', 'Административная работа, секретариат, АХО', 'Банки, инвестиции, лизинг', 'Безопасность, службы охраны', 'Бухгалтерия, финансы, аудит', 'Государственная служба', 'Дизайн', 'Домашний персонал', 'Закупки, снабжение', 'Искусство, культура, развлечения', 'Кадры, управление персоналом', 'Консалтинг, стратегическое развитие', 'Маркетинг, реклама, PR', 'Медицина, фармацевтика, ветеринария', 'Наука, образование, повышение квалификации', 'Некоммерческие организации, волонтерство', 'Обмен персоналом', 'Продажи', 'Промышленность, производство', 'Рабочий персонал', 'Сельское хозяйство', 'Службы доставки', 'СМИ, издательства', 'Спорт, фитнес, салоны красоты, SPA', 'Страхование', 'Строительство, проектирование, недвижимость', 'Сырье', 'Топ-персонал', 'Транспорт, логистика, ВЭД', 'Туризм, гостиницы, общественное питание', 'Услуги, ремонт, сервисное обслуживание', 'Юриспруденция']

Тк мы рассматриваем сферу  it, то нам подойдет отрасль 'IT, Интернет, связь, телеком'. Однако также есть отрасль 'Дизайн'. UX/UI дизайнеры и исследователи тоже относятся к ИТ-профессиям. Надо проверить, возможно, эти ИТ-направления относятся к данной отрасли, а не к 'IT, Интернет, связь, телеком'.


In [93]:
titles_in_design = []
for position in  industry_res[6]['positions']:
    titles_in_design.append(position['title_rus'])
print(titles_in_design)

['Web-дизайн (UI/UX)', 'Архитектура', 'Аудио / Саунд-дизайн', 'Верстка и полиграфия', 'Видео, CGI, анимация', 'Графический дизайн', 'Дизайн интерьера', 'Иллюстрации', 'Ландшафтный дизайн', 'Мода\xa0и\xa0Аксессуары', 'Промышленный дизайн', 'Рекламный дизайн / Ивенты, инсталляции, стенды', 'Фотография', 'Другое', 'Начало карьеры, мало опыта']


И правда - из всех направлений дизайна также подходит направление 'Web-дизайн (UI/UX)' как часть IT-сферы.

In [94]:
industry_res[6]['positions'][0]

{'title_rus': 'Web-дизайн (UI/UX)',
 'url_rus': 'web-dizajn',
 'title': 'Web-дизайн (UI/UX)',
 'id_parent': 62,
 'key': 35}

In [ ]:
# Получаем список вакансий по веб-дизигну с помощью key=35

params = {
    'catalogues': 35,
}

vacs_design = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
design_res = vacs_design.json()
design_res['total']

43

по 40 вакансий на page => 2 pages надо пройти чтобы скачать всё в один датафрейм

In [111]:
res_vacs_design = []
params = {
    'catalogues': 35,
    'count': 40,
    'page': 0
}

for page in range(2):
    params['page']=page
    vacs_design = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
    vacs_design_json = vacs_design.json()
    res_vacs_design.extend(vacs_design_json['objects'])
    
df_design_vacs=pd.DataFrame(res_vacs_design)
df_design_vacs

,contacts_hidden,contact_hiding_cause,canEdit,is_closed,id,id_client,payment_from,payment_to,date_pub_to,date_archived,...,highlight,age_from,age_to,gender,firm_name,firm_activity,link,isBlacklisted,latitude,longitude
0,True,no_resume,False,False,50780033,14215,31500,32000,1775822012,1771476907,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Почта России,Почта России сейчас — это важная часть социаль...,https://bizhbulyak.superjob.ru/vakansii/operat...,False,53.698341,54.268978
1,True,no_resume,False,False,51646102,4944790,0,0,1777646146,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Лаборатория Касперского,Обязанности:Требования:Условия.,https://www.superjob.ru/vakansii/ux-51646102.html,False,NaN,NaN
2,True,no_resume,False,False,51697899,11714,92000,105000,1777802103,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/ux-issledovat...,False,55.764221,37.617321
3,True,no_resume,False,False,51697901,11714,60000,80000,1777802103,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/ux-issledovat...,False,55.764221,37.617321
4,True,no_resume,False,False,51720163,4952362,0,0,1775902509,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Магнит: Старт Карьеры,Обязанности:Требования:Условия.,https://www.superjob.ru/vakansii/kommunikacion...,False,NaN,NaN
5,True,no_resume,False,False,51742606,4954737,0,0,1776769821,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",SKIMS,Обязанности:Требования:Условия.,https://www.superjob.ru/vakansii/dizajner-5174...,False,NaN,NaN
6,True,no_resume,False,False,51782102,2466656,58000,0,1775897834,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}","ОАО ""РЖД""",ПЕРЕВОЗКА ГРУЗОВ И ПАССАЖИРОВ,https://ryazan.superjob.ru/vakansii/brigadir-5...,False,54.631798,39.717808
7,True,no_resume,False,False,51826693,4959764,0,0,1776933780,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Amazon,Обязанности:Требования:Условия.,https://www.superjob.ru/vakansii/stazhjor-ux-d...,False,NaN,NaN
8,True,no_resume,False,False,51836446,14215,31500,32000,1777188133,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Почта России,Почта России сейчас — это важная часть социаль...,https://buraevo.superjob.ru/vakansii/operator-...,False,55.842308,55.404297
9,True,no_resume,False,False,51853120,4957734,0,0,1777804260,0,...,True,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Disney,Обязанности:Требования:Условия.,https://www.superjob.ru/vakansii/stazher-otdel...,False,NaN,NaN


In [ ]:
key_it = 33

res_vacs_it = []
params = {
    'catalogues': key_it,
    'count': 40,
    'page': 0
}
while True:
    
    vacs_it = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
    if vacs_it.status_code != 200:
        break
    vacs_it_json = vacs_it.json()

    if not vacs_it_json['objects']:
        break
    res_vacs_it.extend(vacs_it_json['objects'])
    params['page'] += 1
    

(Источник, где я смотрела способ сбора данных со всех страниц: https://sky.pro/wiki/media/kak-ispolzovat-python-dlya-raboty-s-rest-api/)

In [290]:
df_it = pd.DataFrame(res_vacs_it)
df_it

,contacts_hidden,contact_hiding_cause,canEdit,is_closed,id,id_client,payment_from,payment_to,date_pub_to,date_archived,...,age_from,age_to,gender,firm_name,firm_activity,link,isBlacklisted,latitude,longitude,video
0,True,no_resume,False,False,51697590,11714,0,0,1777812604,1772612101,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://ufa.superjob.ru/vakansii/dezhurnyj-spe...,False,54.783783,56.122543,NaN
1,True,no_resume,False,False,51697719,11714,0,36540,1777799110,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://furmanov.superjob.ru/vakansii/sistemny...,False,57.257141,41.117516,NaN
2,True,no_resume,False,False,51697696,11714,50000,60000,1777798205,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/sistemnyj-adm...,False,46.114944,32.907536,NaN
3,True,no_resume,False,False,51697700,11714,30450,56550,1777798503,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://tosno.superjob.ru/vakansii/sistemnyj-a...,False,59.537891,30.883648,NaN
4,True,no_resume,False,False,51697714,11714,30000,35000,1777799109,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,"Федеральный казённое учреждение, подведомствен...",https://www.superjob.ru/vakansii/sistemnyj-adm...,False,46.114944,32.907536,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,True,no_resume,False,False,51863523,4961470,200000,0,1778151277,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",ПСМ,"Корпорация, целью которой является оптимизация...",https://yakutsk.superjob.ru/vakansii/veduschij...,False,NaN,NaN,NaN
516,True,no_resume,False,False,51863524,4961470,200000,0,1778151277,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",ПСМ,"Корпорация, целью которой является оптимизация...",https://arsenev.superjob.ru/vakansii/veduschij...,False,NaN,NaN,NaN
517,True,no_resume,False,False,51863525,4961470,200000,0,1778151277,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",ПСМ,"Корпорация, целью которой является оптимизация...",https://artem.superjob.ru/vakansii/veduschij-i...,False,NaN,NaN,NaN
518,True,no_resume,False,False,51863526,4961470,200000,0,1778151277,0,...,0,0,"{'id': 0, 'title': 'Не имеет значения'}",ПСМ,"Корпорация, целью которой является оптимизация...",https://bolshoj-kamen.superjob.ru/vakansii/ved...,False,NaN,NaN,NaN


Так как выбранная отрасль - 'IT, Интернет, связь, телеком' содержит не только ИТ сферу, но и работу со связью и телекомом, то стоит подробнее рассмотреть, что необходимо исключить из выборки.

In [289]:
titles_in_it_industry = []
for position in  industry_res[0]['positions']:
    titles_in_it_industry.append(position['title_rus'])
print(titles_in_it_industry)

['AI', 'CRM-системы', 'Data Science', 'DevOps', 'SRE', 'Web-верстка', 'Администрирование баз данных', 'Аналитика', 'Внедрение и сопровождение ПО', 'Игровое ПО / Геймдев', 'Инжиниринг', 'Интернет, создание и поддержка сайтов', 'Информационная безопасность', 'Киберспорт', 'Компьютерная анимация\xa0и\xa0мультимедиа', 'Контент', 'Мобильная разработка', 'Нейросети / Искусственный интеллект', 'Оптимизация, SEO', 'Передача данных и доступ в интернет', 'Разработка и сопровождение банковского ПО', 'Разработка, программирование', 'Сетевые технологии', 'Системная интеграция', 'Системное администрирование', 'Системы автоматизированного проектирования (САПР)', 'Системы управления предприятием (ERP)', 'Сотовые, беспроводные технологии', 'Телекоммуникации и связь', 'Тестирование, QA', 'Техническая документация', 'Техническая поддержка', 'Управление продуктом', 'Управление проектами', 'Управление разработкой', 'Юзабилити', 'Другое', 'Начало карьеры, мало опыта']


Список специализаций отрасли: ['AI', 'CRM-системы', 'Data Science', 'DevOps', 'SRE', 'Web-верстка', 'Администрирование баз данных', 'Аналитика', 'Внедрение и сопровождение ПО', 'Игровое ПО / Геймдев', 'Инжиниринг', 'Интернет, создание и поддержка сайтов', 'Информационная безопасность', 'Киберспорт', 'Компьютерная анимация\xa0и\xa0мультимедиа', 'Контент', 'Мобильная разработка', 'Нейросети / Искусственный интеллект', 'Оптимизация, SEO', 'Передача данных и доступ в интернет', 'Разработка и сопровождение банковского ПО', 'Разработка, программирование', 'Сетевые технологии', 'Системная интеграция', 'Системное администрирование', 'Системы автоматизированного проектирования (САПР)', 'Системы управления предприятием (ERP)', 'Сотовые, беспроводные технологии', 'Телекоммуникации и связь', 'Тестирование, QA', 'Техническая документация', 'Техническая поддержка', 'Управление продуктом', 'Управление проектами', 'Управление разработкой', 'Юзабилити', 'Другое', 'Начало карьеры, мало опыта'].
Здесь нам точно не подойдут: 'Телекоммуникации и связь', 'Техническая поддержка'

Уберем из датафрейма вакансии, которые относятся к этим специализациям

In [301]:
def del_vacs(catalogues, list_specialization):
    for cat in catalogues:
        for pos in cat['positions']:
            if pos['title'] in list_specialization:
                return True
    return False
            
list_specialization = ['Телекоммуникации и связь', 'Телекоммуникации и связь']
mask_del = df_it['catalogues'].apply(lambda x: del_vacs(x, list_specialization))

df_it_right = df_it[~mask_del]
len(df_it_right)

466

После удаления неподходящих специализаций осталось 466 строк.

In [ ]:
params = {
    'keyword': 'IT'
}

vacs = requests.get('https://api.superjob.ru/2.0/vacancies/', headers=headers, params=params)
res = vacs.json()

res.keys()

dict_keys(['objects', 'total', 'more', 'subscription_id', 'subscription_active'])